# Finance Monitoring & Alerting

This notebook implements TPC-H order-total reconciliation against Silver layer tables, aggregates monthly monitoring trends, writes results to a Delta monitoring table, and evaluates alert rules. It concludes with an anomaly simulation that verifies the alerting logic catches injected discrepancies.

## 1. Load Silver Tables

Read the Silver-layer `orders` and `lineitem` Delta tables from `workspace.default_sofiia`.

In [0]:
from pyspark.sql import functions as F

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

# Silver source tables (catalog.schema.table)
ORDERS_TABLE = "workspace.default_sofiia.finance_silver_orders"
LINEITEM_TABLE = "workspace.default_sofiia.finance_silver_lineitem"

# Target monitoring Delta table
MONITORING_TABLE = (
    "workspace.default_sofiia"
    ".finance_monitoring_order_mismatch_monthly_trend"
)

# Reconciliation tolerance — matches TPC-H cent-truncation semantics
TOLERANCE = 0.01

# Anomaly simulation parameters
ANOMALY_COUNT = 5
ANOMALY_OFFSET = 50.0

# ---------------------------------------------------------------------------
# Load Silver tables
# ---------------------------------------------------------------------------

orders = spark.table(ORDERS_TABLE)
lineitem = spark.table(LINEITEM_TABLE)

print("orders schema:")
orders.printSchema()
print("\nlineitem schema:")
lineitem.printSchema()

## 2. TPC-H Order Total Reconciliation

Each line-item total is calculated with cent-level truncation at every step, following the TPC-H specification:

1. `discounted_price = floor(l_extendedprice * (1 - l_discount) * 100) / 100`
2. `line_total = floor(discounted_price * (1 + l_tax) * 100) / 100`

Line totals are summed per order (`l_orderkey`) and compared to `o_totalprice`. A mismatch is flagged when the absolute difference exceeds `$0.01`.

In [0]:
def reconcile_orders(orders_df, lineitem_df, tolerance=TOLERANCE):
    """Reconcile o_totalprice against cent-truncated line-item totals."""
    line_totals = (
        lineitem_df
        .withColumn(
            "discounted_price",
            F.floor(F.col("l_extendedprice") * (1 - F.col("l_discount")) * 100) / 100,
        )
        .withColumn(
            "line_total",
            F.floor(F.col("discounted_price") * (1 + F.col("l_tax")) * 100) / 100,
        )
        .groupBy("l_orderkey")
        .agg(F.sum("line_total").alias("calculated_order_total"))
    )

    reconciliation = (
        orders_df.alias("o")
        .join(line_totals.alias("lt"), F.col("o.o_orderkey") == F.col("lt.l_orderkey"))
        .select(
            F.col("o.o_orderkey"),
            F.col("o.o_totalprice"),
            F.col("o.o_orderdate"),
            F.col("lt.calculated_order_total"),
        )
        .withColumn(
            "difference",
            F.abs(F.col("o_totalprice") - F.col("calculated_order_total")),
        )
        .withColumn("is_mismatched", F.col("difference") > tolerance)
        .withColumn("order_year", F.year(F.col("o_orderdate")))
        .withColumn("order_month", F.month(F.col("o_orderdate")))
        .withColumn("order_year_month", F.date_format(F.col("o_orderdate"), "yyyy-MM"))
    )
    return reconciliation


reconciliation = reconcile_orders(orders, lineitem)
display(reconciliation.orderBy(F.desc("difference")).limit(20))

## 3. Aggregate Monthly Monitoring Trend

Group the reconciliation results by `order_year_month` and compute order counts, mismatch counts, total discrepancy, maximum difference, and mismatch rate percentage.

In [0]:
def aggregate_monthly_trend(reconciliation_df):
    """Aggregate reconciliation results into a monthly monitoring trend."""
    return (
        reconciliation_df
        .groupBy("order_year_month", "order_year", "order_month")
        .agg(
            F.count("o_orderkey").alias("total_orders"),
            F.sum(F.when(F.col("is_mismatched"), 1).otherwise(0)).alias(
                "mismatched_orders_count"
            ),
            F.sum(
                F.when(F.col("is_mismatched"), F.col("difference")).otherwise(0.0)
            ).alias("total_discrepancy_amount"),
            F.max("difference").alias("max_difference"),
        )
        .withColumn(
            "mismatch_rate_pct",
            (F.col("mismatched_orders_count") / F.col("total_orders")) * 100,
        )
        .orderBy("order_year_month")
    )


monthly_trend = aggregate_monthly_trend(reconciliation)
display(monthly_trend)

# Summary statistics across all months
summary_stats = monthly_trend.agg(
    F.sum("total_orders").alias("total_orders_all_months"),
    F.sum("mismatched_orders_count").alias("total_mismatched_orders"),
    F.sum("total_discrepancy_amount").alias("total_discrepancy_amount"),
    F.max("max_difference").alias("max_difference_overall"),
    F.max("mismatch_rate_pct").alias("max_mismatch_rate_pct"),
)
display(summary_stats)

## 4. Write to Delta Table

Persist the monthly trend as a managed Delta table for downstream monitoring dashboards and scheduled alert evaluation.

In [0]:
(
    monthly_trend.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(MONITORING_TABLE)
)
print(f"Monthly trend written to {MONITORING_TABLE}")

## 5. Alert Evaluation

Evaluate the alert rule: if any mismatched orders exist across all months, raise an ALERT; otherwise report OK.

In [0]:
total_mismatched = summary_stats.first()["total_mismatched_orders"] or 0

if total_mismatched == 0:
    print("OK")
else:
    print(f"ALERT: {total_mismatched} mismatched orders detected")

## 6. Anomaly Simulation & Alert Verification

Artificially inject a `+$50.00` error into `o_totalprice` for 5 orders, re-run the full reconciliation pipeline, and verify that the alert rule catches exactly 5 mismatches and triggers the ALERT status.

In [0]:
# Select 5 order keys to perturb
anomaly_keys = [
    row.o_orderkey for row in orders.select("o_orderkey").limit(5).collect()
]
print(f"Selected order keys for anomaly injection: {anomaly_keys}")

# Create modified orders with +50.0 added to selected orders
modified_orders = orders.withColumn(
    "o_totalprice",
    F.when(
        F.col("o_orderkey").isin(anomaly_keys),
        F.col("o_totalprice") + ANOMALY_OFFSET,
    ).otherwise(F.col("o_totalprice")),
)

# Re-run reconciliation with perturbed orders
anomaly_reconciliation = reconcile_orders(modified_orders, lineitem)
anomaly_monthly = aggregate_monthly_trend(anomaly_reconciliation)

# Count total mismatches across all months
anomaly_total_mismatched = anomaly_monthly.agg(
    F.sum("mismatched_orders_count").alias("total")
).first()["total"]

print(f"\nAnomaly simulation results:")
print(f"  Mismatched orders detected: {anomaly_total_mismatched}")
print(f"  Expected:                   {ANOMALY_COUNT}")

# Evaluate the alert rule
if anomaly_total_mismatched == 0:
    print("\nStatus: OK")
else:
    print(f"\nStatus: ALERT: {anomaly_total_mismatched} mismatched orders detected")

# Show months with mismatches
display(anomaly_monthly.filter(F.col("mismatched_orders_count") > 0))

# Verify exactly 5 mismatches
assert anomaly_total_mismatched == ANOMALY_COUNT, (
    f"Expected {ANOMALY_COUNT} mismatches, got {anomaly_total_mismatched}"
)
print(
    f"\nVerification passed: exactly {ANOMALY_COUNT} mismatches detected "
    f"and the ALERT status was triggered."
)